# TERRA embeddings on TENX99

Goal: compare three linear-probe baselines for gene expression prediction on a
subset of TENX99 cells — TERRA embeddings alone, UNI2-h embeddings alone, and
their concatenation — to test whether TERRA (spatial transcriptomics FM) and
UNI2-h (pathology FM) carry complementary information.

In [ ]:
from pathlib import Path

import os
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix

HEST_DIR = Path("/lustre/scratch126/cellgen/lotfollahi/so25/data/hest_samples")
NOTEBOOK_DIR = Path(
    "/lustre/scratch126/cellgen/lotfollahi/so25/projects/cellbridge/notebooks/exploration"
)
DATA_DIR = Path("/lustre/scratch126/cellgen/lotfollahi/so25/data/")
EMB_DIR = DATA_DIR / "embeddings"
CACHE_DIR = DATA_DIR / "terra_cache"
EMB_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_ID = "TENX99"
RANDOM_STATE = 42
N_CELLS = 10_000
MIN_TOTAL_COUNTS = 50
QV_THRESHOLD = 20  # Xenium's standard Q-Score cutoff; 10x's cell x gene matrix counts only qv >= 20 calls

plt.rcParams["figure.dpi"] = 110

### Select cell subset

We read all the columns from the 126.9M-row transcripts file and filter in three steps:

1. drop **unassigned** transcripts (`cell_id == -1`, Xenium's convention for
   calls not attributed to any cell);
2. drop **low-confidence** calls (`qv < 20`) — this is the same cutoff 10x uses
   to build its standard cell x gene matrix, so the reconstructed counts match
   what TERRA was trained/validated on (see the `qv` analysis in notebook 1);
3. drop Xenium's **negative-control** probes (`NegControlProbe_*`,
   `NegControlCodeword_*`, `BLANK_*`).

We then aggregate transcript counts per (cell, gene) into a sparse cells x
genes matrix. Spatial coordinates are the mean **native Xenium micron** position
(`x_location`/`y_location`) per cell — the coordinate system TERRA was trained
on — and the mean **H&E-pixel** position (`he_x`/`he_y`) is kept in `obs` for
later image work. `cell_id` is an explicit column here, so no
positional-index assumption is needed (unlike joining against `xenium_seg`'s
geometry-only parquet).

In [ ]:
def is_control_probe(name):
    return name.startswith(("NegControlProbe_", "NegControlCodeword_", "BLANK_"))


transcripts = pd.read_parquet(
    HEST_DIR / "transcripts" / f"{SAMPLE_ID}_transcripts.parquet",
    # columns=["cell_id", "feature_name", "x_location", "y_location", "he_x", "he_y", "qv"],
)
print(f"Raw transcript rows: {len(transcripts):,}")

In [ ]:
transcripts.iloc[0]

In [ ]:
# Step 1 filtering
transcripts = transcripts[transcripts["cell_id"] != -1]
print(f"After dropping unassigned (cell_id == -1): {len(transcripts):,}")

# Step 2 filtering
transcripts = transcripts[transcripts["qv"] >= QV_THRESHOLD]
print(f"After dropping low-confidence calls (qv < {QV_THRESHOLD}): {len(transcripts):,}")

# Step 3 filtering
transcripts = transcripts[
    ~transcripts["feature_name"].astype(str).map(is_control_probe)
]
print(f"After dropping negative-control probes: {len(transcripts):,}")

In [ ]:
cell_centroids = transcripts.groupby("cell_id")[
    ["x_location", "y_location", "he_x", "he_y"]
].mean()
cell_centroids.columns = ["x_centroid", "y_centroid", "he_x", "he_y"]
print(f"Centroids computed for {len(cell_centroids):,} cells")

In [ ]:
transcripts

In [ ]:
# Aggregate transcript counts into a sparse (cells x genes) matrix
cell_cat = pd.Categorical(transcripts["cell_id"])
gene_cat = pd.Categorical(transcripts["feature_name"])

counts = (
    coo_matrix(
        (np.ones(len(transcripts), dtype=np.float32), (cell_cat.codes, gene_cat.codes))
    ).tocsr()
)  # duplicate (cell, gene) coordinates are summed on conversion -> transcript counts

adata_full = ad.AnnData(
    X=counts,
    obs=pd.DataFrame(index=[str(c) for c in cell_cat.categories]),
    var=pd.DataFrame(index=list(gene_cat.categories)),
)
adata_full.obs["cell_id"] = adata_full.obs_names
# Align both coordinate systems to the matrix's cell (row) order.
cent = cell_centroids.loc[cell_cat.categories]
# .obsm['spatial'] = native Xenium microns -> used for the TERRA neighbor graph.
adata_full.obsm["spatial"] = cent[["x_centroid", "y_centroid"]].to_numpy()
# H&E-pixel centroids kept in obs for later image-patch extraction (UNI2-h side).
adata_full.obs["he_x"] = cent["he_x"].to_numpy()
adata_full.obs["he_y"] = cent["he_y"].to_numpy()

sc.pp.calculate_qc_metrics(adata_full, inplace=True, percent_top=None)
print(adata_full)

### Extract TERRA embeddings

In [ ]:
import yaml
from terra import download_pretrained, harmonize_adata, tokenize_adata, embed_dataset
from datasets import load_from_disk

model_dir = Path(download_pretrained("Lotfollahi-lab/TERRA-112M"))
print("model_dir:", model_dir)

with open(model_dir / "model_config.yaml") as f:
    model_config = yaml.safe_load(f)
print("n_neighs:", model_config["data"]["n_neighs"])
print("embedding dim (enc_emb_dim):", model_config["meta"]["enc_emb_dim"])
print("tokenizer_type:", model_config["data"]["tokenizer_type"])

### k=10 spatial neighbor graph on the full section

Built with the same call TERRA's own tokenizer uses internally
(`squidpy.gr.spatial_neighbors`, `coord_type='generic'`, `n_neighs=10`) on the
**native Xenium micron** coordinates in `obsm['spatial']`, over **all ~890k
cells** in `adata_full` — not the 10,000-cell subsample — so neighborhoods
reflect real tissue context. Note the tokenizer *recomputes* this graph itself
from `obsm['spatial']`. This is just for visualization.

In [ ]:
sq.gr.spatial_neighbors(
    adata_full, coord_type="generic", spatial_key="spatial", n_neighs=10
)
print(
    adata_full.obsp["spatial_connectivities"].shape,
    "edges:",
    adata_full.obsp["spatial_connectivities"].nnz,
)

In [ ]:
# Visualize the k=10 graph in a small 150x150 um window (native pixel size
# 0.2125 um/px), drawn directly on top of the H&E patch it corresponds to.
# The edges come from obsp (built on native microns); we draw the nodes at
# their H&E-pixel positions (obs['he_x'/'he_y']) so they line up with the H&E
# patch -- obsm['spatial'] is microns, not pixels.
import tifffile
import zarr

pixel_size = 0.2125
window_um = 150
window_px = int(round(window_um / pixel_size))

xy = adata_full.obs[["he_x", "he_y"]].to_numpy()  # H&E pixel centroids
center = np.median(xy, axis=0)
x0 = int(center[0] - window_px / 2)
y0 = int(center[1] - window_px / 2)

in_window = (
    (xy[:, 0] >= x0)
    & (xy[:, 0] < x0 + window_px)
    & (xy[:, 1] >= y0)
    & (xy[:, 1] < y0 + window_px)
)
idx_in_window = np.where(in_window)[0]

wsi_store = tifffile.imread(
    HEST_DIR / "wsis" / f"{SAMPLE_ID}.tif", aszarr=True, level=0
)
wsi_level0 = zarr.open(wsi_store, mode="r")
patch = wsi_level0[
    y0 : y0 + window_px, x0 : x0 + window_px
]  # zarr axes are (row=y, col=x, channel)

conn = adata_full.obsp["spatial_connectivities"].tocsr()

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(patch)
for i in idx_in_window:
    neighbors = conn[i].indices
    for j in neighbors:
        if in_window[j]:
            ax.plot(
                [xy[i, 0] - x0, xy[j, 0] - x0],
                [xy[i, 1] - y0, xy[j, 1] - y0],
                c="black",
                linewidth=0.7,
                zorder=1,
            )
ax.scatter(
    xy[idx_in_window, 0] - x0, xy[idx_in_window, 1] - y0, s=10, c="yellow", zorder=2
)
ax.axis("off")
ax.set_title(
    f"k=10 spatial neighbor graph on H&E ({window_um}x{window_um} um, {len(idx_in_window)} cells)"
)
fig.tight_layout()

### Harmonize and tokenize the full section

`harmonize_adata` maps our real gene symbols to the model's Ensembl-ID
vocabulary and validates raw integer counts; `tokenize_adata` builds each
cell's tokenized neighborhood (recomputing the `n_neighs=10` graph from
`obsm['spatial']`) for **every** cell in the full section.

In [ ]:
gene_mapping_dict_file_path = model_dir / "ensembl_dictionary.pkl"
gene_occurrence_count_file_path = model_dir / "gene_count_dictionary.pkl"
assert gene_mapping_dict_file_path.is_file()
assert gene_occurrence_count_file_path.is_file()

harmonized_full = harmonize_adata(
    adata_full,
    gene_mapping_dict_file_path=str(gene_mapping_dict_file_path),
    gene_occurrence_count_file_path=str(gene_occurrence_count_file_path),
    min_cells_per_gene=10,
    min_genes_per_cell=10,
)
print(harmonized_full)
print(f"Genes kept after harmonization: {harmonized_full.n_vars} / {adata_full.n_vars}")
print(f"Cells kept after harmonization: {harmonized_full.n_obs} / {adata_full.n_obs}")

In [ ]:
sampled_obs = harmonized_full.obs.sample(n=N_CELLS, random_state=RANDOM_STATE)
sampled_cell_ids = sampled_obs["cell_id"].to_numpy()
assert len(set(sampled_cell_ids)) == N_CELLS
print(f"Sampled exactly {len(sampled_cell_ids):,} cells (random_state={RANDOM_STATE})")

In [ ]:
# Distribution of total_counts in the sampled subset
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(
    harmonized_full.obs["total_counts"], bins=60, color="lightgray", label="harmonized + filtered cells"
)
axes[0].hist(
    sampled_obs["total_counts"],
    bins=60,
    color="steelblue",
    alpha=0.7,
    label="sampled 10k",
)
axes[0].set_title("total_counts: harmonized population vs sampled subset")
axes[0].set_xlabel("total_counts")
axes[0].legend()

axes[1].hist(sampled_obs["total_counts"], bins=60, color="steelblue")
axes[1].axvline(
    sampled_obs["total_counts"].median(), color="black", linestyle="--", label="median"
)
axes[1].set_title("Sampled 10k: total_counts distribution")
axes[1].set_xlabel("total_counts")
axes[1].legend()
fig.tight_layout()

print(sampled_obs["total_counts"].describe())

In [ ]:
# Save the selected cell IDs now, independent of the embedding step below
np.save(EMB_DIR / f"{SAMPLE_ID}_sampled_cell_ids.npy", sampled_cell_ids)
print(
    f"Saved {len(sampled_cell_ids):,} cell IDs -> {EMB_DIR / f'{SAMPLE_ID}_sampled_cell_ids.npy'}"
)

In [ ]:
import logging

logging.basicConfig(level="INFO")

FULL_TOKENIZED_PATH = DATA_DIR / f"{SAMPLE_ID}_full_tokenized.dataset"

if FULL_TOKENIZED_PATH.exists():
    full_dataset = load_from_disk(str(FULL_TOKENIZED_PATH))
    print(f"Loaded cached tokenized dataset ({len(full_dataset):,} cells) from {FULL_TOKENIZED_PATH}")
else:
    full_dataset = tokenize_adata(
        adata=harmonized_full,
        model_folder_path=str(model_dir),
        cache_directory_path=str(CACHE_DIR),
        nproc=len(os.sched_getaffinity(0)),
    )
    full_dataset.save_to_disk(str(FULL_TOKENIZED_PATH))
    print(f"Tokenized and cached {len(full_dataset):,} cells -> {FULL_TOKENIZED_PATH}")

print(full_dataset)

In [ ]:
sampled_ids_str = set(map(str, sampled_cell_ids))
keep_idx = [
    i for i, cid in enumerate(full_dataset["cell_id"]) if str(cid) in sampled_ids_str
]
assert len(keep_idx) == N_CELLS, f"expected {N_CELLS} matches, got {len(keep_idx)}"

subset_dataset = full_dataset.select(keep_idx)
subset_dataset

In [ ]:
embeddings = embed_dataset(
    dataset=subset_dataset,
    model_folder_path=str(model_dir),
    batch_size=32,
)
print(list(embeddings))
for key, arr in embeddings.items():
    print(key, arr.shape)

### Save embeddings

We save both `cell_emb` and `neighborhood_emb` (whichever `terra-uni2` fusion
step follows can pick the right one) alongside the exact cell IDs in the row
order `embed_dataset` produced. We also store each cell's **H&E-pixel centroid**
(`he_x`/`he_y`, level-0 px) in that same row order.

In [ ]:
ordered_cell_ids = np.array(subset_dataset["cell_id"])

ordered_ids_str = [str(c) for c in ordered_cell_ids]
he = harmonized_full.obs.loc[ordered_ids_str, ["he_x", "he_y"]]
assert list(he.index) == ordered_ids_str, "he centroid row order mismatch"

np.savez(
    EMB_DIR / f"{SAMPLE_ID}_terra.npz",
    cell_ids=ordered_cell_ids,
    cell_emb=embeddings["cell_emb"],
    neighborhood_emb=embeddings["neighborhood_emb"],
    he_x=he["he_x"].to_numpy(),
    he_y=he["he_y"].to_numpy(),
)
print(f"Saved -> {EMB_DIR / f'{SAMPLE_ID}_terra.npz'}")
print("cell_emb shape:", embeddings["cell_emb"].shape)
print("neighborhood_emb shape:", embeddings["neighborhood_emb"].shape)
print("he_x/he_y saved for", len(he), "cells (aligned to embedding order)")

### Are the embeddings informative?

A very high average cosine similarity (~1) means the embeddings collapsed. Then UMAP both embeddings,
colored by `total_counts` and by 2-3 top highly-variable genes.

In [ ]:
for emb_key in ["cell_emb", "neighborhood_emb"]:
    X = embeddings[emb_key]
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
    n = Xn.shape[0]
    S = Xn.sum(axis=0)
    avg_cos = (S @ S - n) / (n * (n - 1))
    print(f"{emb_key}: average cosine similarity = {avg_cos:.3f}")

In [ ]:
# Build an AnnData for the 10k embedded cells, in the exact row order that
# embed_dataset produced (ordered_cell_ids), so the obsm embeddings line up
# with X. Lets us UMAP each embedding and color by total_counts / markers.
ordered_ids_str = [str(c) for c in ordered_cell_ids]

adata_sub = harmonized_full[ordered_ids_str].copy()
assert list(adata_sub.obs["cell_id"]) == ordered_ids_str, "row order mismatch"

adata_sub.obsm["X_terra_cell"] = embeddings["cell_emb"]
adata_sub.obsm["X_terra_neighborhood"] = embeddings["neighborhood_emb"]

print(adata_sub)

In [ ]:
for emb_key in ["X_terra_cell", "X_terra_neighborhood"]:
    sc.pp.neighbors(adata_sub, use_rep=emb_key, key_added=emb_key)
    sc.tl.umap(adata_sub, neighbors_key=emb_key)
    sc.pl.umap(adata_sub, neighbors_key=emb_key, title=emb_key)

In [ ]:
sc.tl.leiden(adata_sub, neighbors_key="neighborhood_emb", key_added="niche",
             resolution=0.6, flavor="igraph", n_iterations=2, directed=False)
sc.tl.leiden(adata_sub, neighbors_key="cell_emb", key_added="cell_type",
             resolution=0.6, flavor="igraph", n_iterations=2, directed=False)

In [ ]:
sc.tl.umap(adata_sub, neighbors_key="neighborhood_emb")
sc.pl.umap(adata_sub, neighbors_key="neighborhood_emb", color="niche",
           palette="tab20", title="Niches")

sc.tl.umap(adata_sub, neighbors_key="cell_emb")
sc.pl.umap(adata_sub, neighbors_key="cell_emb", color="cell_type",
           palette="tab20b", title="Cell types")

If the UMAPs above show clear structure (cells organizing by total counts
and/or marker gene expression, not a uniform blob), TERRA's embeddings are
informative for this sample.